In [1]:
import os
import numpy as np
from ase import io
from ase.atoms import Atoms

add an argument (say distort_axes) that tells the function which lattice directions are allowed to change (e.g. "xy", "xz", "yz", or "xyz"). Then we apply:

hydrostatic scaling only on those axes (so "xy" scales a and b but leaves c alone — perfect for slabs)

random symmetric strain/shear only within that subspace (so for "xy" you get only xx/yy/xy components; no xz/yz, no zz)

In [2]:
def make_strained_rattled_ensembles(
    atoms_or_filename,
    scalings,
    n_per_scaling=2,
    rattle_stdev=0.01,
    epsmax=0.05,
    seed=None,
    out_dir=None,
    vasp_direct=True,
    vasp_sort=False,
    distort_axes="xyz",   # e.g. "xy" for slabs, "xyz" for bulk
):
    """
    Generate ensembles with:
      1) scaling applied only along specified axes
      2) additional random symmetric strain/shear within those axes
      3) atomic rattling

    distort_axes:
      - "xy"  -> affect x/y only (no z, no xz/yz shear)
      - "xz"  -> affect x/z only
      - "yz"  -> affect y/z only
      - "xyz" -> full 3D (bulk-like)
    """

    rng = np.random.default_rng(seed)

    # Load structure
    if isinstance(atoms_or_filename, Atoms):
        base = atoms_or_filename.copy()
    else:
        base = io.read(atoms_or_filename)

    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)

    axes_map = {"x": 0, "y": 1, "z": 2}
    ax = sorted({axes_map[c] for c in distort_axes.lower() if c in axes_map})
    if len(ax) == 0:
        raise ValueError(f"distort_axes must contain at least one of 'x','y','z' (got {distort_axes!r})")

    generated = []

    for s in scalings:
        for k in range(n_per_scaling):
            atoms = base.copy()

            cell = atoms.get_cell().array

            # (1) Scaling only along selected axes (scales chosen lattice vectors)
            scale_vec = np.ones(3)
            for i in ax:
                scale_vec[i] = float(s)
            cell_scaled = (cell.T * scale_vec).T  # scale row-vectors i (a,b,c)
            atoms.set_cell(cell_scaled, scale_atoms=True)

            # (2) Random symmetric strain/shear restricted to selected axes
            E = np.zeros((3, 3))
            for i in ax:
                for j in ax:
                    if j < i:
                        continue
                    val = rng.uniform(-epsmax, epsmax)
                    E[i, j] = val
                    E[j, i] = val

            D = np.eye(3) + E

            cell2 = atoms.get_cell().array
            new_cell = (D @ cell2.T).T
            atoms.set_cell(new_cell, scale_atoms=True)

            # (3) Rattle
            atoms.rattle(
                stdev=rattle_stdev,
                seed=int(rng.integers(0, 2**31 - 1))
            )

            generated.append(atoms)

            # Write structure
            if out_dir is not None:
                axes_tag = "".join("xyz"[i] for i in ax)
                folder_name = f"s{int(round(s*1000)):04d}_{axes_tag}_k{k:03d}"
                folder_path = os.path.join(out_dir, folder_name)
                os.makedirs(folder_path, exist_ok=True)

                io.write(
                    os.path.join(folder_path, "POSCAR"),
                    atoms,
                    format="vasp",
                    direct=vasp_direct,
                    sort=vasp_sort
                )

    return generated

In [11]:
! pwd

/nfshome/sicolo/work/MLIP/surfaces


In [6]:
scalings = [
    0.95, 0.96, 0.97, 0.98, 0.99, 1.00, 1.01, 1.02, 1.03, 1.04, 1.05
    ]

structures = make_strained_rattled_ensembles(
    atoms_or_filename="NiO/111/O-term/POSCAR.NiO.111.p2x2_O-terminated.vasp",
    scalings=scalings,
    distort_axes="xy",            # important for slabs!
    n_per_scaling=2,
    rattle_stdev=0.01,
    epsmax=0.05,
    seed=42,                      # optional but good for reproducibility
    out_dir="NiO/111/O-term/."
)